# Sle control gene summary


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

PERMUTATION_CSV = "results/permutation_full/sle_vs_hc/gene_permutation_pvalues.csv"
PEPTIDE_LOGIT_CSV = "results/donor_level_results_strict/donor_strict_peptide_logistic_OR.csv"
PEPTIDE_DISC_CSV = "results/donor_level_results_strict/donor_strict_peptide_discrete_OR.csv"
OUTPUT_CSV = "results/permutation_full/sle_vs_hc/slehc_gene_level_clean_summary.csv"

STATS_TO_INCLUDE = ['fisher', 'max_absZ']

def main():

    print("[1] Loading peptide-level data...")
    pep_logit = pd.read_csv(PEPTIDE_LOGIT_CSV)
    pep_disc = pd.read_csv(PEPTIDE_DISC_CSV)
    print(f"    Logistic: {len(pep_logit):,} peptides")
    print(f"    Discrete: {len(pep_disc):,} peptides")

    if 'pep_short' not in pep_logit.columns:
        raise ValueError("pep_short column not found in logistic OR CSV. "
                         "Check that the file includes gene annotation columns.")

    print(f"    Logistic columns: {list(pep_logit.columns)}")

    pep_merged = pep_logit.merge(
        pep_disc[['peptide', 'n_high_case_donors', 'n_high_control_donors',
                  'prop_high_case_donors', 'prop_high_control_donors']],
        on='peptide', how='left'
    )

    print("[2] Aggregating to gene level (min-p peptide per gene)...")

    abs_z = stats.norm.ppf(1 - pep_merged['p_value_two_sided'].clip(1e-300, 1) / 2)
    pep_merged['z_score'] = np.sign(pep_merged['log_or']) * abs_z

    idx_min = pep_merged.groupby('gene')['p_value_two_sided'].idxmin()
    gene_effects = pep_merged.loc[idx_min].copy()

    n_pep_per_gene = pep_merged.groupby('gene').size().rename('n_peptides')
    gene_effects = gene_effects.merge(n_pep_per_gene, left_on='gene', right_index=True, how='left')

    gene_effects = gene_effects.rename(columns={
        'pep_short': 'best_pep_short',
        'peptide': 'best_peptide',
        'p_value_two_sided': 'best_peptide_pval',
    })

    cols_to_keep = [
        'gene',
        'n_peptides',
        'best_pep_short',
        'best_peptide',
        'log_or',
        'z_score',
        'odds_ratio',
        'best_peptide_pval',
        'prop_high_case_donors',
        'prop_high_control_donors',
        'n_high_case_donors',
        'n_high_control_donors',
    ]
    cols_to_keep = [c for c in cols_to_keep if c in gene_effects.columns]
    gene_effects = gene_effects[cols_to_keep].reset_index(drop=True)

    print(f"    {len(gene_effects):,} genes")

    print("[3] Loading permutation results...")
    perm_df = pd.read_csv(PERMUTATION_CSV)
    print(f"    {len(perm_df):,} genes with permutation p-values")

    print("[4] Merging...")
    merged = gene_effects.merge(perm_df, on='gene', how='inner')
    print(f"    {len(merged):,} genes after merge")

    final_cols = [
        'gene',
        'n_peptides',
        'best_pep_short',
        'best_peptide',
        'log_or',
        'z_score',
        'odds_ratio',
        'best_peptide_pval',
        'prop_high_case_donors',
        'prop_high_control_donors',
    ]

    for stat in STATS_TO_INCLUDE:
        obs_col = f'{stat}_obs'
        perm_col = f'{stat}_perm_p'
        if obs_col in merged.columns:
            final_cols.append(obs_col)
        if perm_col in merged.columns:
            final_cols.append(perm_col)

    print("[5] Computing BH FDR q-values...")
    for stat in STATS_TO_INCLUDE:
        perm_col = f'{stat}_perm_p'
        qval_col = f'{stat}_qval'
        if perm_col in merged.columns:
            pvals = merged[perm_col].values
            mask = np.isfinite(pvals)
            merged[qval_col] = np.nan
            if mask.sum() > 0:
                _, qvals, _, _ = multipletests(pvals[mask], method='fdr_bh')
                merged.loc[merged.index[mask], qval_col] = qvals
            final_cols.append(qval_col)

            n_sig_01 = (merged[qval_col] <= 0.1).sum()
            n_sig_005 = (merged[qval_col] <= 0.05).sum()
            print(f"    {stat}: FDR<0.1 = {n_sig_01}, FDR<0.05 = {n_sig_005}")

    final_cols = [c for c in final_cols if c in merged.columns]
    clean_df = merged[final_cols].copy()

    sort_col = 'truncsum_0050_perm_p' if 'truncsum_0050_perm_p' in clean_df.columns else 'fisher_perm_p'
    clean_df = clean_df.sort_values(sort_col).reset_index(drop=True)

    print(f"\n[6] Saving to {OUTPUT_CSV}")
    clean_df.to_csv(OUTPUT_CSV, index=False)

    print(f"\nShape: {clean_df.shape}")
    print(f"\nColumns ({len(clean_df.columns)}):")
    for col in clean_df.columns:
        print(f"  - {col}")

    print(f"\nTop 15 genes by {sort_col}:")
    preview_cols = ['gene', 'best_pep_short', 'n_peptides', 'log_or', 'z_score',
                    'fisher_obs', 'fisher_perm_p', 'fisher_qval',
                    'max_absZ_obs', 'max_absZ_perm_p', 'max_absZ_qval']
    preview_cols = [c for c in preview_cols if c in clean_df.columns]
    print(clean_df[preview_cols].head(15).to_string(index=False))

    key_genes = ['SNRNP70', 'YTHDF2', 'SSB', 'TROVE2', 'SNRPB', 'SNRPD1']
    key_df = clean_df[clean_df['gene'].isin(key_genes)]
    if len(key_df) > 0:
        print(f"\nKey SLE autoantigens:")
        print(key_df[preview_cols].to_string(index=False))

    return clean_df

if __name__ == '__main__':
    clean_df = main()